# RAG Playground (Reranking Branch)

Dieses Notebook ist fuer den `reranking`-Branch vorbereitet.

Es deckt drei Teststufen ab:
- Retrieval-Vergleich fuer `bm25`, `dense` und `hybrid`
- optionales manuelles Reranking auf bereits geholten Dokumenten
- optionaler End-to-End-RAG-Lauf mit oder ohne Reranker


In [ ]:
# Setup: Projekt-Root robust setzen
import os
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)

print('cwd:', cwd)
print('ROOT:', ROOT)
print('Working dir:', Path.cwd())
print('Python:', sys.executable)


In [ ]:
# Imports + Projektstatus
import json
import time
from statistics import mean

from src.config import RAW_DIR, CHROMA_DIR, DEFAULT_LLM_MODEL
from src.indexing import build_index, load_index, clear_retrieval_caches
from src.rag import get_docs_for_query, answer_with_rag_mode, DEFAULT_RERANKER_MODEL

RERANKER_AVAILABLE = True
rerank_documents = None
reranker_error = None
try:
    from src.reranking import rerank_documents
except Exception as exc:
    RERANKER_AVAILABLE = False
    reranker_error = exc

pdfs = sorted(RAW_DIR.rglob('*.pdf')) if RAW_DIR.exists() else []
print('RAW_DIR:', RAW_DIR)
print('CHROMA_DIR:', CHROMA_DIR)
print('PDF count:', len(pdfs))
for p in pdfs[:10]:
    print('-', p.name)
if len(pdfs) > 10:
    print(f'... ({len(pdfs) - 10} more)')
print('Index dir exists:', CHROMA_DIR.exists())
print('Default LLM model:', DEFAULT_LLM_MODEL)
print('Default reranker model:', DEFAULT_RERANKER_MODEL)
print('Reranker available:', RERANKER_AVAILABLE)
if reranker_error is not None:
    print('Reranker import error:', repr(reranker_error))


In [ ]:
# Konfiguration fuer Tests
QUESTION_ID = 'q1'
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200

REBUILD_INDEX = False
FORCE_REBUILD_INDEX = False

RETRIEVAL_K = 10
RETRIEVAL_RUNS = 2
RETRIEVAL_MODES = ('bm25', 'dense', 'hybrid')

RUN_MANUAL_RERANK = True
MANUAL_RERANK_MODE = 'hybrid'
MANUAL_RERANK_K = 20
MANUAL_RERANK_TOP_N = 3

RUN_RAG = True
RAG_MODE = 'hybrid'
RAG_K = 20           # Immer hoch lassen: Reranker braucht viele Kandidaten

# Fragetyp steuert RERANK_TOP_N:
#   'focused'      -> 3  (einfache/konkrete Fragen, 1 Stelle im Dokument)
#   'normal'       -> 5  (Standard)
#   'cross_domain' -> 7  (Fragen ueber mehrere Kapitel/Themen)
QUERY_TYPE = 'focused'

_TOP_N_MAP = {'focused': 3, 'normal': 5, 'cross_domain': 7}
RERANK_TOP_N = _TOP_N_MAP[QUERY_TYPE]
USE_RERANKER = True

print('QUESTION_ID:', QUESTION_ID)
print('Chunking:', CHUNK_SIZE, CHUNK_OVERLAP)
print('Rebuild index:', REBUILD_INDEX, 'force:', FORCE_REBUILD_INDEX)
print('Retrieval test:', RETRIEVAL_MODES, 'k=', RETRIEVAL_K, 'runs=', RETRIEVAL_RUNS)
print('Manual rerank:', RUN_MANUAL_RERANK, MANUAL_RERANK_MODE, MANUAL_RERANK_K, MANUAL_RERANK_TOP_N)
print(f'RAG: mode={RAG_MODE}, k={RAG_K}, query_type={QUERY_TYPE}, rerank_top_n={RERANK_TOP_N}')

In [ ]:
# Fragen zentral laden
QUESTIONS_PATH = ROOT / 'data' / 'eval' / 'questions.jsonl'

def load_questions(path=QUESTIONS_PATH):
    items = {}
    with Path(path).open('r', encoding='utf-8-sig') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            qid = str(obj['id']).strip().lstrip('\ufeff')
            items[qid] = obj['query']
    return items

questions = load_questions()
print('Questions file:', QUESTIONS_PATH)
print('Verfuegbare IDs:', ', '.join(sorted(questions.keys())))

query = questions[QUESTION_ID]
print('Ausgewaehlte ID:', QUESTION_ID)
print('Frage:', query)


In [ ]:
# Index laden oder bei Bedarf neu bauen
if REBUILD_INDEX:
    print('Rebuilding index ...')
    vs = build_index(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        force_rebuild=FORCE_REBUILD_INDEX,
    )
    print('Index rebuilt.')
else:
    vs = load_index()
    print('Existing index loaded.')

# Optional nach Code-/Datenaenderungen:
# clear_retrieval_caches()


In [ ]:
# Helpers: Retrieval-Benchmark + Anzeige
def _preview(text: str, n: int = 240) -> str:
    text = ' '.join((text or '').split())
    return text[:n] + ('...' if len(text) > n else '')

def show_docs(docs, title='Docs'):
    print(f'\n=== {title} | count={len(docs)} ===')
    for i, d in enumerate(docs, start=1):
        md = d.metadata or {}
        print(f"{i}. source={md.get('source')}, page={md.get('page')}, section={md.get('section', '?')}")
        print('   ' + _preview(d.page_content))

def benchmark_retrieval(query: str, modes=('bm25', 'dense', 'hybrid'), k: int = 5, runs: int = 3, chunk_size: int = 1200, chunk_overlap: int = 200):
    print(f'Query: {query}')
    print(f'Modes={modes}, k={k}, runs={runs}, chunk_size={chunk_size}, chunk_overlap={chunk_overlap}')
    print('-' * 90)
    results = {}

    for mode in modes:
        latencies = []
        last_docs = []
        for idx in range(runs):
            t0 = time.perf_counter()
            docs = get_docs_for_query(
                query=query,
                mode=mode,
                k=k,
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
            )
            dt = time.perf_counter() - t0
            latencies.append(dt)
            last_docs = docs
            print(f'[{mode}] run {idx + 1}/{runs}: {dt:.3f}s | docs={len(docs)}')

        avg_s = mean(latencies) if latencies else float('nan')
        results[mode] = {'latencies': latencies, 'avg_s': avg_s, 'docs': last_docs}
        print(f'[{mode}] avg: {avg_s:.3f}s')
        print('-' * 90)

    print('\nTop-Chunks aus letztem Lauf je Modus:')
    for mode in modes:
        show_docs(results[mode]['docs'], title=mode.upper())

    return results


In [ ]:
# Retrieval-Vergleich
retrieval_results = benchmark_retrieval(
    query=query,
    modes=RETRIEVAL_MODES,
    k=RETRIEVAL_K,
    runs=RETRIEVAL_RUNS,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)


In [ ]:
# Optional: manuelles Reranking auf Retrieval-Kandidaten
if not RUN_MANUAL_RERANK:
    print('RUN_MANUAL_RERANK=False -> diese Stufe wird uebersprungen.')
elif not RERANKER_AVAILABLE:
    print('Reranker ist in dieser Umgebung nicht nutzbar.')
    print('Fehler:', repr(reranker_error))
else:
    docs = get_docs_for_query(
        query=query,
        mode=MANUAL_RERANK_MODE,
        k=MANUAL_RERANK_K,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    print(f'Vor Reranking: {len(docs)} Dokumente')
    show_docs(docs[:min(5, len(docs))], title=f'{MANUAL_RERANK_MODE.upper()} BEFORE')

    reranked_docs = rerank_documents(
        query=query,
        docs=docs,
        top_n=MANUAL_RERANK_TOP_N,
    )
    show_docs(reranked_docs, title='RERANKED')


In [ ]:
# Optional: End-to-End RAG-Lauf
if RUN_RAG:
    effective_reranker = USE_RERANKER and RERANKER_AVAILABLE
    t0 = time.perf_counter()
    rag_res = answer_with_rag_mode(
        query=query,
        mode=RAG_MODE,
        k=RAG_K,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        use_reranker=effective_reranker,
        rerank_top_n=RERANK_TOP_N,
    )
    dt = time.perf_counter() - t0

    print(f'RAG mode={RAG_MODE} total={dt:.3f}s | reranker={effective_reranker}')
    print('\nAntwort:\n', rag_res['answer'])
    print('\nQuellen:')
    for s in rag_res.get('sources', []):
        print(f"- {s.get('source')} (Seite {s.get('page')})")
else:
    print('RUN_RAG=False -> nur Retrieval ausgefuehrt.')

In [ ]:
from pathlib import Path
import json

p = ROOT / "data" / "eval" / "questions.jsonl"

rows = [
    {"id": "q1", "query": "Welche Füll- und Lösezeiten gelten für den Bremszylinder in der Bremsstellung G und P nach UIC 540?"},
    {"id": "q2", "query": "Welche Komponenten verursachten laut Peche die meisten außerplanmäßigen Instandhaltungsmaßnahmen an Bremsanlagen?"},
    {"id": "q3", "query": "Wie viele Maßnahmenvorschläge und Handlungsfelder wurden im DZSF-Bericht identifiziert?"},
    {"id": "q4", "query": "Welche Sensitivitätsindizes nach Sobol' werden in der Dissertation von Finke verwendet?"},
    {"id": "q5", "query": "Welche relevanten Normen werden bei der Entwicklung der automatisierten Bremsprobe nach Peche genannt?"},
    {"id": "q6", "query": "Welche drei Entwicklungen im Eisenbahnwesen nennt Finke als potenzielle Einflussfaktoren auf die Längsdynamik von Güterzügen?"},
    {"id": "q7", "query": "Welche Zuglänge in Metern und welche Art der Bremsung werden im Prüfszenario 4 zur Plausibilitätsprüfung des Längsdynamik-Gesamtmodells verwendet?"},
    {"id": "q8", "query": "Warum wird bei der Überwachung des Bremsgestängestellers die Rotationsdetektion gegenüber Dehnmessstreifen bevorzugt?"},
    {"id": "q9", "query": "Welche Methode verwendet Finke zur globalen Sensitivitätsanalyse und warum werden Pseudozufallszahlen statt echter Zufallszahlen verwendet?"},
    {"id": "q10", "query": "Welches Ziel verfolgt das DZSF-Forschungsprojekt zu sensorbasierten Technologien im Bahnsystem?"},
    {"id": "q11", "query": "Wie wird bei Peche der Schwellwert zur Bestimmung der Bremsstellung im Feldversuch berechnet?"},
    {"id": "q12", "query": "Welchen Einfluss hat die Bremszylinderfüllzeit laut Finke auf die Längsdynamik des Güterzugs?"},
    {"id": "q13", "query": "Aus welchem physikalischen bzw. mechanischen Grund ist es bei sehr geringen Bremszylinderdrücken kaum möglich, durch die Messung der Gestängesteller-Rotation zuverlässig auf die Bremsstellung zu schließen?"},
    {"id": "q14", "query": "Wieso wird bei stichprobenbasierten Sensitivitätsanalysen häufig der Einsatz von Low-Discrepancy-Sequenzen anstelle von einfachen Pseudozufallszahlen empfohlen?"},
    {"id": "q15", "query": "Welche Rolle spielt die Digitale Automatische Kupplung (DAK) sowohl bei Finke als auch im DZSF-Bericht?"},
    {"id": "q16", "query": "Wie ergänzen sich die Ansätze von Peche (Bremsgestängestellerüberwachung) und der DZSF-Bericht (sensorbasierte Technologien) hinsichtlich der automatischen Bremsprobe?"},
    {"id": "q17", "query": "Welche Gemeinsamkeiten gibt es zwischen den Herausforderungen der Sensortechnologien im DZSF-Bericht und den Anforderungen an die Bremsanlage im Schienengüterverkehr bei Peche und Finke?"},
    {"id": "q18", "query": "Welche Bremssystemparameter sind sowohl in Finkes Längsdynamik-Analyse als auch in Peches Überwachungssystem relevant?"},
    {"id": "q19", "query": "Welche Sensortypen, die im DZSF-Bericht beschrieben werden, könnten für die von Finke untersuchten Längsdynamik-Parameter eingesetzt werden?"},
    {"id": "q20", "query": "Welche Bedeutung hat die Interoperabilität im Schienengüterverkehr laut Peche und wie spiegelt sich dieses Thema im DZSF-Bericht wider?"},
    {"id": "q21", "query": "Inwiefern ergänzen sich das Simulationsmodell von Finke (ELSA) und die sensorbasierten Ansätze aus dem DZSF-Bericht?"},
    {"id": "q22", "query": "Aus welchem Jahr stammt der DZSF-Bericht über sensorbasierte Technologien und wer sind die Autoren?"},
    {"id": "q23", "query": "Worum geht es bei Finke und welche drei Entwicklungen motivieren die Arbeit?"},
    {"id": "q24", "query": "Was ist Peches Hauptziel und wo wird es erläutert?"},
    {"id": "q25", "query": "In welche fünf Kapitel gliedert sich der DZSF-Bericht?"},
    {"id": "q26", "query": "Welche Anforderungen stellt die DIN 5566-2 an Führerräume von Schienenfahrzeugen?"},
    {"id": "q27", "query": "Wie hoch ist die zulässige Höchstgeschwindigkeit für Güterzüge auf deutschen Hauptstrecken?"},
    {"id": "q28", "query": "Welche konkreten Testergebnisse aus den Messkampagnen von Peche zeigen, dass das System in 100% der Fälle korrekt funktioniert?"},
    {"id": "q29", "query": "Welche Bremsleistung in kW gibt Finke als optimalen Wert für Güterzüge an?"},
    {"id": "q30", "query": "Welche Ergebnisse liefert der Vergleich zwischen dem Bremsgestängesteller-Überwachungssystem von Peche und einem vergleichbaren System von Siemens?"}
]

with p.open("w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Wrote:", p)
print("Size:", p.stat().st_size)
